# SDF for quadratic bezier

$F_{c_1, c_2, c_3}: p_s \to \{\tilde{t}, \tilde{p}, d^2, s\}$

- $c_1, c_2, c_3$ — control points of bezier
- $p_s$ — sample point in UV
- $\tilde{t}$ — param along curve
- $\tilde{p}$ — closest point on curve, $ = S(\tilde{t})$
- $d^2$ — squared distance, $= ||p_s - \tilde{p}||^2$
- $s = ±1$ — side from curve direction

Nearly-flat bezier:

- dropping z for distance calculations
- restoring z for bumps


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
from typing import NamedTuple
import numpy as np
from numpy.random import random as nprand
from numpy.typing import NDArray
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, unflat, f32, disquance, normalize

In [ ]:
def pladd(plot, *args):
    for a in args:
        plot.__iadd__(a)


zero3 = arrgs(0.0, 0.0, 0.0)

# Bezier

Centered around $c2$ and $t \in [-0.5, +0.5]$

- $c_2 \to 0$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axis
- $v_d = v_3 - v_1$ — parabolic direction
- $c_a = .25 v_a$ — apex

$$
S(t') - c_2 =
\big[1, t', t'^2 \big]
\begin{bmatrix}
c_a \\
v_d \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{bmatrix}
v_d \\
2 v_a \\
\end{bmatrix}
$$


In [ ]:
class Bezielt:
    """Centered at c_2 and t=-0.5..+0.5"""

    c2: npvec
    ca: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    def __init__(self, c1: npvec, c2: npvec, c3: npvec):
        self.c2 = c2
        self.v1 = c1 - c2
        self.v3 = c3 - c2
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1
        self.ca = 0.25 * self.va

    def loc(self, point: npvec) -> npvec:
        return point - self.c2

    def glb(self, point: npvec) -> npvec:
        return point + self.c2

    def point(self, t: float) -> npvec:
        return t * t * self.va + t * self.vd + self.ca

    def flow(self, t: float) -> npvec:
        return 2 * self.va * t + self.vd

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

# The SDF

$p_c = c_a - p$ ­— relative to apex

$$
[1, t, t^2, t^3]
\begin{pmatrix}
p_c·v_d \\
2 p_c·v_a + v_d·v_d\\
3 v_a·v_d \\
2 v_a·v_a \\
\end{pmatrix}
=0
$$


In [ ]:
def solve_np(bezier: Bezielt, p: npvec) -> tuple[float, ...]:
    va = bezier.va[:2]
    vd = bezier.vd[:2]
    pc = bezier.ca[:2] - p[:2]

    aa = float(va @ va)
    ad = float(va @ vd)
    dd = float(vd @ vd)
    pa = float(pc @ va)
    pd = float(pc @ vd)

    a_3 = 2 * aa
    a_2 = 3 * ad
    a_1 = dd + 2 * pa
    a_0 = pd

    roots = np.roots((a_3, a_2, a_1, a_0))
    return tuple(r.real for r in roots if r.imag == 0.0)

In [ ]:
from math import sqrt, cbrt, cos, acos, pi


def solve(bezier: Bezielt, p: npvec) -> tuple[float, ...]:
    va = bezier.va[:2]
    vd = bezier.vd[:2]
    pc = bezier.ca[:2] - p[:2]

    aa = float(va @ va)
    ad = float(va @ vd)
    dd = float(vd @ vd)
    pa = float(pc @ va)
    pd = float(pc @ vd)

    a_1 = dd + 2 * pa
    p3 = (2 * aa * a_1 - 3 * ad**2) / (12 * aa**2)
    q2 = (ad**3 - aa * ad * a_1 + 2 * aa**2 * pd) / (8 * aa**3)
    off = ad / (2 * aa)

    D = p3**3 + q2**2

    if D > 0:
        C = cbrt(sqrt(D) - q2)
        t_ = C - p3 / C
        return (t_ - off,)
    else:
        rt = sqrt(-p3)
        k = 2 * rt
        ph = 2 * pi / 3
        th = acos(q2 / (p3 * rt)) / 3
        t_0 = k * cos(th)
        t_1 = k * cos(th - ph)
        t_2 = k * cos(th - 2 * ph)
        return (t_0 - off, t_1 - off, t_2 - off)

In [ ]:
class Projection(NamedTuple):
    t: float
    pnt: npvec
    dsq: float = 0
    sgn: int = 0

    @property
    def sdist(self):
        return self.sgn * np.sqrt(self.dsq)

In [ ]:
def side(bezier: Bezielt, t: float, sample: npvec) -> int:
    ort = bezier.loc(sample) - bezier.point(t)
    tng = bezier.flow(t)
    crz = ort[1] * tng[0] - ort[0] * tng[1]
    return int(np.sign(crz))

In [ ]:
def project(bezier: Bezielt, sample: npvec) -> Projection:
    ps = bezier.loc(sample)
    tt = solve(bezier, ps)
    if len(tt) == 1:
        t = tt[0]
        pnt = bezier.point(t)
        dsq = disquance(pnt, ps)
    else:
        pnts = tuple(bezier.point(t) for t in tt)
        dsqs = tuple(disquance(p, ps) for p in pnts)
        best = np.argmin(dsqs)
        t = tt[best]
        pnt = pnts[best]
        dsq = dsqs[best]

    sgn = side(bezier, t, sample)

    if -0.5 > t or t > +0.5:
        return Projection(t, bezier.glb(pnt), np.inf, sgn)

    return Projection(t, bezier.glb(pnt), dsq, sgn)


---


In [ ]:
def random_points():
    p2 = (nprand(3) - 0.5) * arrgs(0.5, 0.5, 0)
    p1 = (nprand(3) - 0.5) * arrgs(1.0, 1.0, 0)
    p3 = (nprand(3) - 0.5) * arrgs(1.0, 1.0, 0)
    return arrgs(p1, p2, p3)

In [ ]:
controls = arr(((-0.5, 0.5, 0.0), (0.0, 0.0, 0.0), (0.5, 0.5, 0.0)))
bezier = Bezielt(controls[0], controls[1], controls[2])
tspace = np.linspace(-0.5, +0.5, 16, dtype=np.float32)

# Plot


In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-1.0, -1.0, 0.0, 1.0, 1.0, 0.5),
    grid_auto_fit=False,
    mode="callback",
)
plot.layout = wg.Layout(width="720px", height="720px")
# plot.camera_auto_fit = False
# plot.camera = [0, 0, 5, 0, 1, 0, 0, 0, 0]

In [ ]:
randomize_btn = wg.Button(description="randomize")
sample_btn = wg.Button(description="sample")
toggle0 = wg.Checkbox(description="controls", value=False)
toggle1 = wg.Checkbox(description="tangents", value=False)
toggle2 = wg.Checkbox(description="texture", value=False)

In [ ]:
wg.HBox([plot, wg.VBox([toggle0, toggle1, toggle2, randomize_btn, sample_btn])], layout=dict(width="100%", grid_gap="8px"))

In [ ]:
k3controls = k3d.line(vertices=controls, color=0x808080, line_width=0.125, shader="thick", visible=toggle0.value)
k3curve = k3d.line(vertices=[], shader="mesh", color=0xF0F0F0, line_width=0.25, color_map=k3d.colormaps.matplotlib_color_maps.Rainbow, color_range=[-0.5, +0.5])
k3flow = k3d.vectors(origins=[(0, 0, 0)], vectors=[(0, 0, 0)], use_head=False, color=0x000000, head_color=0xF0F0F0, visible=False)
k3points = k3d.points(positions=[], shader="mesh", point_size=0.03125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-0.5, +0.5])
k3proj = k3d.line(vertices=[], attribute=[], shader="thick", line_width=0.125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-0.5, +0.5])


pladd(plot, k3proj, k3points, k3controls, k3curve, k3flow)

In [ ]:
def toggle(obj, val):
    obj.visible = val


toggle0.observe(lambda ch: toggle(k3controls, ch.new), "value")
toggle1.observe(lambda ch: toggle(k3flow, ch.new), "value")

In [ ]:
def regenerate_curve(bezier: Bezielt):
    k3curve.vertices = garr(bezier.glb(p) for p in bezier.curve(tspace))
    k3curve.attribute = tspace
    k3flow.origins = garr(bezier.glb(bezier.point(t)) for t in (-0.5, 0.0, +0.5))
    k3flow.vectors = garr(normalize(bezier.flow(t)) for t in (-0.5, 0.0, +0.5))
    k3flow.visible = toggle1.value


regenerate_curve(bezier)

In [ ]:
def randomize():
    global bezier
    controls[:] = random_points()
    k3controls.vertices = f32(controls)
    bezier = Bezielt(*controls)
    regenerate_curve(bezier)
    regenerate_image(bezier)

In [ ]:
randomize_btn.on_click(lambda _: randomize())

In [ ]:
def sample_random():
    sample = unflat(nprand(2) - 0.5)
    proj = project(bezier, sample)
    k3proj.vertices = [sample, proj.pnt]
    k3proj.attribute = [proj.sdist, proj.sdist]
    k3points.positions = [sample, proj.pnt]
    k3points.attribute = [proj.sdist, proj.sdist]


sample_btn.on_click(lambda btn: sample_random())

## Texture


In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(-0.5, 0.5, PIX), np.arange(-0.5, 0.5, PIX))
COORDS = np.stack((Y, X)).T.reshape((RES * RES, 2)) + 0.5 * PIX  # pixel centers

In [ ]:
imagedata = np.random.random((RES, RES))
k3image = k3d.texture(attribute=imagedata, interpolation=False, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, 1.0])

pladd(plot, k3image)

In [ ]:
toggle2.observe(lambda ch: toggle(k3image, ch.new), "value")

In [ ]:
def regenerate_image(bezier: Bezielt):
    global imagedata
    imagedata = garr(project(bezier, unflat(s)).sdist for s in COORDS)
    k3image.attribute = imagedata.reshape((RES, RES))


regenerate_image(bezier)